# Sujet 07, Santander Transaction : chargement, contrôle et plancher
**Auteur : [votre nom] · Équipe :Othiniel PADONOU · Date : 09/11/26**
Porteur : 2 
**Périmètre de cette partie** : charger et compter les données, vérifier les types, chercher les valeurs impossibles, mesurer le plancher (règle sans modèle).

**Métrique du projet** : AUC (aire sous la courbe ROC)
**Données** : train.csv (160 000 clients, avec `target`), a_decider.csv (40 000 clients, sans `target`)

# Santander Transaction -Sujet 7 
**Séance : **01 
**NOM : **Othiniel PADONOU 
**Porteur : **P2


**Etapes assignées : ** 
1-2 : Charger et comptes 
1-3 : Typer 
1-5 : Valeur Impossible 
1-10 : Le plancher 

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42
CIBLE = "target"
DOSSIER = Path(".")

print("prêt")


prêt


# Charger et faires les comptes 
ici il s'agit , de charger les données et de faires les comptes c'est a dire verifier si les données  dont  l'enoncé  à parler sont Juste .


In [14]:
train = pd.read_csv(DOSSIER / "train.csv" , engine="pyarrow")
decider = pd.read_csv(DOSSIER / "a_decider.csv", engine="pyarrow")

print("train   :", train.shape)
print("adecider  :",decider.shape) 

print("attendu    : (160000, 202) et (40000, 201)")

vars_ = [c for c in train.columns if c.startswith("var_")]
print("nombre de variables :", len(vars_))

print("\nID en double (train)     :", train["ID_code"].duplicated().sum())
print("ID en double (a_decider) :", decider["ID_code"].duplicated().sum())
print("ID communs aux deux      :", len(set(train["ID_code"]) & set(decider["ID_code"])))

print("\nPositifs :", int(train[CIBLE].sum()), f"({train[CIBLE].mean():.2%})")

train   : (160000, 202)
adecider  : (40000, 201)
attendu    : (160000, 202) et (40000, 201)
nombre de variables : 200

ID en double (train)     : 0
ID en double (a_decider) : 0
ID communs aux deux      : 0

Positifs : 16078 (10.05%)


# TYPER  
**Nous devons verifier le type de chaque colonne ,

In [15]:
#verif contient 200 lignes. Personne ne les lit toutes, et un problème (une valeur bizarre) peut se cacher dans une seule. 
# Le top trie le tableau et ne garde que les N premières lignes : 
# celles qui ont le plus de chances d’être anormales.
#  1.3 : ce que pandas croit vs ce que les colonnes contiennent

verif = pd.DataFrame({
    # ce que pandas croit 
    "type_pandas": train[vars_].dtypes,
    # part de la valeur la plus répété : si elle est élevee , c'est une marque d'absence possible
    "part_valeur_frequente": train[vars_].apply(
        lambda s: s.value_counts(normalize=True).iloc[0]),
    #extrème : repéré un -999 ou un 999ç isolé 
    "min": train[vars_].min(),
    "max": train[vars_].max(),
})
# vue d'ensemble des 200 variables 
print("Vue d'ensemble sur verif ")
display(verif.describe())
#les 5 variables ou les mm valeurs reviennent le plus 
print("Résumées des statistiques des 200  variables ")
display(verif.sort_values("part_valeur_frequente", ascending=False).head(5))

print("ID_code : type", train["ID_code"].dtype)
print("target  : type", train[CIBLE].dtype,
      "| valeurs", sorted(train[CIBLE].unique())) 

# controle : chaque liste doit etre vide, et la cible ne doit valoir que 0 ou 1 
non_decimales = [c for c in vars_ if train[c].dtype.kind != "f"]   # type differents de décimal
sentinelles = verif[verif["part_valeur_frequente"] > 0.01].index.tolist()  # valeur répétée sur plus de 1 des lignes 
cible_ok = set(train[CIBLE].unique()) <= {0, 1} # la cible est binaires 


#Verdict
print("\n--- VERDICT 1.3 ---")
print("variables non décimales           :", non_decimales)
print("variables avec valeur répétée >1% :", sentinelles)
print("target ne vaut que 0 ou 1         :", cible_ok)

if not (non_decimales or sentinelles) and cible_ok:
    print("=> aucune colonne à changer de type.")   # verification , p  ndas avaut raisin 
else:
    print("=> À EXAMINER : voir les listes non vides.")

Vue d'ensemble sur verif 


,part_valeur_frequente,min,max
count,200.000000,200.000000,200.000000
mean,0.000125,-8.400716,21.796640
std,0.000421,15.098920,13.827085
min,0.000031,-90.252500,1.571900
25%,0.000050,-16.028525,12.567000
50%,0.000059,-5.368400,18.412750
75%,0.000081,2.152025,27.767925
max,0.005556,13.729000,74.032100


Résumées des statistiques des 200  variables 


,type_pandas,part_valeur_frequente,min,max
var_68,float64,0.005556,4.9938,5.0469
var_126,float64,0.001544,11.2654,15.5109
var_108,float64,0.001531,13.7290,14.7430
var_12,float64,0.001063,13.4346,14.6545
var_91,float64,0.000356,6.5218,7.4895


ID_code : type str
target  : type int64 | valeurs [np.int64(0), np.int64(1)]

--- VERDICT 1.3 ---
variables non décimales           : []
variables avec valeur répétée >1% : []
target ne vaut que 0 ou 1         : True
=> aucune colonne à changer de type.


#  Valeur Impossible 
Chercher les valeurs qui ne peuvent pas exister dans la réalité : un âge négatif, un montant impossible, une valeur qui sert de marque d’absence.
Tu produis. La liste des valeurs trouvées, et le traitement que tu leur appliques.

In [16]:
# 1.5 : valeurs impossibles

X = train[vars_]

# Valeurs infinies et manquantes
print("valeurs infinies   :", int(np.isinf(X.to_numpy()).sum()))
print("valeurs manquantes :", int(X.isna().sum().sum()))

# Plage globale des 200 variables (l'énoncé annonce -73,8 à +65,4)
print("minimum global :", float(X.min().min()))
print("maximum global :", float(X.max().max()))

# Valeurs sentinelles classiques présentes dans les données
sentinelles_connues = [-999, -9999, 9999, 999, -1]
for v in sentinelles_connues:
    n = int((X == v).sum().sum())
    print(f"nombre de cellules égales à {v} :", n)

# Cible hors de 0 et 1
print("cible hors {0, 1} :", int((train[CIBLE].isin([0, 1])).sum()))

# Variables dont les valeurs sont toutes identiques
print("variables constantes :", [c for c in vars_ if X[c].nunique() == 1])

valeurs infinies   : 0
valeurs manquantes : 0
minimum global : -90.2525
maximum global : 74.0321
nombre de cellules égales à -999 : 0
nombre de cellules égales à -9999 : 0
nombre de cellules égales à 9999 : 0
nombre de cellules égales à 999 : 0
nombre de cellules égales à -1 : 78
cible hors {0, 1} : 160000
variables constantes : []


# Le Plancher 
. Construis une règle qui n’utilise aucune variable, et mesure son score. C’est le score que ton modèle devra battre.
Tu produis. La règle choisie, la valeur qu’elle produit, et son score.

In [17]:
# 1.10 : plancher = règle sans aucune variable
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

y = train[CIBLE]

# Découpage : on mesure sur une partie de train (25 %), stratifié pour garder 10 % de positifs
_, y_val = train_test_split(y, test_size=0.25, stratify=y, random_state=SEED) #découpage de train en 25pourcent
print("taux de positifs en validation :", round(y_val.mean(), 4)) #arrondi à 4chiffres apres la virgule 

# Règle 1 : la même valeur pour tous les clients (le taux de positifs de train)
taux = y.mean()
score_constant = np.full(len(y_val), taux) #fabrication d'un liste remplie de la mm valeur 
auc_constant = roc_auc_score(y_val, score_constant)

# Règle 2 : un score tiré au hasard
rng = np.random.default_rng(SEED) # cree un generateur aleatoire initialisées 
auc_hasard = roc_auc_score(y_val, rng.random(len(y_val)))

print(f"Règle 1 (constante = {taux:.4f}) : AUC = {auc_constant:.4f}")
print(f"Règle 2 (hasard)               : AUC = {auc_hasard:.4f}")

taux de positifs en validation : 0.1005
Règle 1 (constante = 0.1005) : AUC = 0.5000
Règle 2 (hasard)               : AUC = 0.5077
